# Session 8 · Working with Hive and Impala

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-08-hive-and-impala.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

**Setup:** the Spark examples need PySpark (and Java, which Google Colab already has).

In [1]:
%pip install -q pyspark

### Example 1 · Managed vs external tables

In [2]:
import os, tempfile
from pyspark.sql import SparkSession
base = tempfile.mkdtemp()
spark = (SparkSession.builder.master("local[2]").appName("hive-tables")
         .config("spark.sql.warehouse.dir", os.path.join(base, "warehouse"))
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

ext_dir = os.path.join(base, "landing", "cities")
spark.createDataFrame([("Karachi", "Sindh"), ("Lahore", "Punjab"), ("Quetta", "Balochistan")],
                      ["city", "province"]).write.mode("overwrite").parquet(ext_dir)

wh = spark.conf.get("spark.sql.warehouse.dir").replace("file:", "")
spark.sql("DROP TABLE IF EXISTS cities_managed"); spark.sql("DROP TABLE IF EXISTS cities_external")
spark.sql("CREATE TABLE cities_managed (city STRING, province STRING) USING parquet")
spark.sql("INSERT INTO cities_managed SELECT * FROM parquet.`%s`" % ext_dir)
spark.sql(f"CREATE TABLE cities_external USING parquet LOCATION '{ext_dir}'")
spark.sql("SELECT * FROM cities_external ORDER BY city").show()

for t in ["cities_managed", "cities_external"]:
    kind = spark.sql(f"DESCRIBE TABLE EXTENDED {t}").filter("col_name = 'Type'").first()["data_type"]
    print(f"{t:<16} type = {kind}")

managed_dir = os.path.join(wh, "cities_managed")
spark.sql("DROP TABLE cities_managed"); spark.sql("DROP TABLE cities_external")
print("\nafter DROP TABLE:")
print("  managed files still exist? ", os.path.exists(managed_dir))
print("  external files still exist?", os.path.exists(ext_dir))

+-------+-----------+
|   city|   province|
+-------+-----------+
|Karachi|      Sindh|
| Lahore|     Punjab|
| Quetta|Balochistan|
+-------+-----------+

cities_managed   type = MANAGED


cities_external  type = EXTERNAL

after DROP TABLE:
  managed files still exist?  False
  external files still exist? True


### Example 2 · Partitioning and partition pruning

In [3]:
import os, tempfile
from pyspark.sql import SparkSession, functions as F
base = tempfile.mkdtemp()
spark = (SparkSession.builder.master("local[2]").appName("partitions")
         .config("spark.sql.warehouse.dir", os.path.join(base, "warehouse"))
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

sales = (spark.range(0, 6000)
         .withColumn("year", F.when(F.col("id") % 2 == 0, 2025).otherwise(2026))
         .withColumn("month", (F.floor(F.col("id") / 2) % 12 + 1).cast("int"))
         .withColumn("city", F.element_at(F.array(*[F.lit(c) for c in ["Karachi", "Lahore", "Islamabad"]]),
                                          (F.floor(F.col("id") / 24) % 3 + 1).cast("int")))
         .withColumn("amount", (F.col("id") * 37 % 5000 + 100).cast("double")))

sales.write.partitionBy("year", "month").mode("overwrite").saveAsTable("sales")
spark.sql("SHOW PARTITIONS sales").show(5)
tdir = os.path.join(spark.conf.get("spark.sql.warehouse.dir").replace("file:", ""), "sales")
print("year folders:", sorted(d for d in os.listdir(tdir) if d.startswith("year")))
print("month folders in year=2026:", len(os.listdir(os.path.join(tdir, "year=2026"))))

q = spark.sql("SELECT city, SUM(amount) AS total FROM sales WHERE year = 2026 AND month = 10 GROUP BY city ORDER BY city")
q.show()
plan = q._jdf.queryExecution().executedPlan().toString()
import re
print(re.sub(r"#\d+", "", re.search(r"PartitionFilters: \[[^\]]*\]", plan).group(0)))
print("total partitions:", spark.sql("SHOW PARTITIONS sales").count(), "| folders scanned: year=2026/month=10 only")

+------------------+
|         partition|
+------------------+
| year=2025/month=1|
|year=2025/month=10|
|year=2025/month=11|
|year=2025/month=12|
| year=2025/month=2|
+------------------+
only showing top 5 rows
year folders: ['year=2025', 'year=2026']
month folders in year=2026: 12


+---------+--------+
|     city|   total|
+---------+--------+
|Islamabad|209649.0|
|  Karachi|219156.0|
|   Lahore|215945.0|
+---------+--------+

PartitionFilters: [isnotnull(year), isnotnull(month), (year = 2026), (month = 10)]


total partitions: 24 | folders scanned: year=2026/month=10 only


### Example 3 · Bucketing and table sampling

In [4]:
import os, tempfile
from pyspark.sql import SparkSession, functions as F
base = tempfile.mkdtemp()
spark = (SparkSession.builder.master("local[2]").appName("buckets")
         .config("spark.sql.warehouse.dir", os.path.join(base, "warehouse"))
         .config("spark.sql.autoBroadcastJoinThreshold", "-1")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # force a real join so bucketing matters
orders = spark.range(0, 20000).select((F.col("id") % 500).alias("customer_id"), (F.col("id") % 977).alias("amount"))
custs = spark.range(0, 500).select(F.col("id").alias("customer_id"), (F.col("id") % 5).alias("segment"))
orders.write.bucketBy(8, "customer_id").sortBy("customer_id").mode("overwrite").saveAsTable("orders_b")
custs.write.bucketBy(8, "customer_id").sortBy("customer_id").mode("overwrite").saveAsTable("customers_b")

wh = spark.conf.get("spark.sql.warehouse.dir").replace("file:", "")
files = [f for f in os.listdir(os.path.join(wh, "orders_b")) if f.endswith(".parquet")]
print("orders_b data files:", len(files))
j = spark.table("orders_b").join(spark.table("customers_b"), "customer_id")
plan = j._jdf.queryExecution().executedPlan().toString()
print("shuffle needed for join:", "Exchange hashpartitioning" in plan)
print("rows joined:", j.count())
spark.sql("SELECT segment, COUNT(*) AS orders, SUM(amount) AS revenue FROM orders_b JOIN customers_b USING (customer_id) GROUP BY segment ORDER BY segment").show()

orders_b data files: 16


shuffle needed for join: False


rows joined: 20000


+-------+------+-------+
|segment|orders|revenue|
+-------+------+-------+
|      0|  4000|1928034|
|      1|  4000|1928126|
|      2|  4000|1928218|
|      3|  4000|1928310|
|      4|  4000|1928402|
+-------+------+-------+



### Example 4 · Complex types, explode and window functions

In [5]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder.master("local[2]").appName("hiveql")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

spark.createDataFrame(
    [("Ali", "Karachi", ["hadoop", "spark"], 82), ("Sara", "Lahore", ["hive"], 91),
     ("Omar", "Karachi", ["spark", "hive", "pig"], 77), ("Hina", "Lahore", ["hadoop"], 88),
     ("Zain", "Karachi", ["pig"], 95)],
    "name string, city string, skills array<string>, score int").createOrReplaceTempView("students")

spark.sql("""
  SELECT skill, COUNT(*) AS students
  FROM students LATERAL VIEW explode(skills) s AS skill
  GROUP BY skill ORDER BY students DESC, skill
""").show()

spark.sql("""
  SELECT city, name, score,
         RANK() OVER (PARTITION BY city ORDER BY score DESC) AS rank_in_city,
         ROUND(AVG(score) OVER (PARTITION BY city), 1)     AS city_avg
  FROM students ORDER BY city, rank_in_city
""").show()

+------+--------+
| skill|students|
+------+--------+
|hadoop|       2|
|  hive|       2|
|   pig|       2|
| spark|       2|
+------+--------+



+-------+----+-----+------------+--------+
|   city|name|score|rank_in_city|city_avg|
+-------+----+-----+------------+--------+
|Karachi|Zain|   95|           1|    84.7|
|Karachi| Ali|   82|           2|    84.7|
|Karachi|Omar|   77|           3|    84.7|
| Lahore|Sara|   91|           1|    89.5|
| Lahore|Hina|   88|           2|    89.5|
+-------+----+-----+------------+--------+

